In [8]:
import pandas as pd

In [9]:
df_jun = pd.read_csv('../data/raw/listings_jun.csv', encoding='utf-8')
df_jun.shape

(30259, 90)

In [10]:
df_sep = pd.read_csv('../data/raw/listings_sep.csv', encoding='utf-8')
df_sep.shape

(29918, 90)

In [11]:
df_mar = pd.read_csv('../data/raw/listings_mar.csv', encoding='utf-8')
df_mar.shape

(35979, 90)

In [20]:
df_mar['id'].duplicated().sum()

np.int64(0)

In [22]:
df_all = pd.concat([df_sep, df_jun, df_mar], axis = 0, ignore_index=True).copy()

In [33]:
df_all[df_all['id'].duplicated()].groupby(df_all['id'])

In [34]:
df_all.loc[df_all['id'].duplicated()].sort_values('id').head(20)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
29918,2539,https://www.airbnb.com/rooms/2539,20260614073253,2026-06-15,city scrape,11 Min to Manhattan • Prospect Park • Fast WiFi,"Bright, serene room in a renovated apartment h...",NaN,https://a0.muscache.com/pictures/hosting/Hosti...,2787,...,5.00,4.78,4.78,NaN,NaN,5,0,5,0,0.08
29919,6848,https://www.airbnb.com/rooms/6848,20260614073253,2026-06-14,city scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,4.80,4.69,4.59,NaN,NaN,1,1,0,0,0.95
60184,6848,https://www.airbnb.com/rooms/6848,20260316030535,2026-03-16,city scrape,Only 2 stops to Manhattan studio,Comfortable studio apartment with super comfor...,NaN,https://a0.muscache.com/pictures/e4f031a7-f146...,15991,...,4.80,4.69,4.59,NaN,NaN,1,1,0,0,0.96
29920,6872,https://www.airbnb.com/rooms/6872,20260614073253,2026-06-14,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,5.00,5.00,5.00,NaN,NaN,2,0,2,0,0.04
60185,6872,https://www.airbnb.com/rooms/6872,20260316030535,2026-03-16,city scrape,Uptown Sanctuary w/ Private Bath (Month to Month),This charming distancing-friendly month-to-mon...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,16104,...,5.00,5.00,5.00,NaN,NaN,2,0,2,0,0.04
29921,6990,https://www.airbnb.com/rooms/6990,20260614073253,2026-06-14,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,4.94,4.85,4.84,NaN,NaN,1,0,1,0,1.24
60186,6990,https://www.airbnb.com/rooms/6990,20260316030535,2026-03-16,city scrape,UES Beautiful Blue Room,Beautiful peaceful healthy home,NaN,https://a0.muscache.com/pictures/45fb4ec7-6856...,16800,...,4.94,4.85,4.84,NaN,NaN,1,0,1,0,1.26
29922,7097,https://www.airbnb.com/rooms/7097,20260614073253,2026-06-14,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,4.93,4.95,4.82,NaN,NaN,2,0,2,0,2.12
60187,7097,https://www.airbnb.com/rooms/7097,20260316030535,2026-03-16,city scrape,"Perfect for Your Parents, With Garden & Patio",Parents/grandparents coming to town or are you...,NaN,https://a0.muscache.com/pictures/aaac19fc-4b4d...,17571,...,4.93,4.95,4.82,NaN,NaN,2,0,2,0,2.15
29923,7801,https://www.airbnb.com/rooms/7801,20260614073253,2026-06-14,city scrape,Sunny Williamsburg Loft with Private Cedar Sauna,A huge loft in a repurposed factory building i...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,21207,...,4.80,5.00,4.90,NaN,NaN,1,1,0,0,0.06


## Requirements for data cleaning
`host_identity_verified` should be t,
`price_quote_price_per_night` should not be NA,
`has_availability` should be t,
active referring to `estimated_occupancy_l365d` not 0 or `availability_365` (availability for next 365 days) not 0




In [35]:
df_all.shape

(96156, 90)

In [36]:
verified = df_all['host_identity_verified'] == 't'
has_price = df_all['price_quote_price_per_night'].notna()
available = df_all['has_availability'] == 't'
active = ((df_all['estimated_occupancy_l365d'] != 0) | (df_all['availability_365'] != 0))

df_clean = df_all[verified & has_price & available & active].copy()
df_clean.shape

(59874, 90)

In [37]:
df_clean.loc[df_clean['maximum_nights'] < df_clean['minimum_nights'], ['accommodates', 'beds', 'bedrooms']]

,accommodates,beds,bedrooms


In [38]:
df_clean['review_scores_rating'].describe()

count    43892.000000
mean         4.723619
std          0.467889
min          1.000000
25%          4.670000
50%          4.860000
75%          5.000000
max          5.000000
Name: review_scores_rating, dtype: float64

In [39]:
df_analysis = df_clean[['id', 'name', 'host_id', 'host_name', 'host_is_superhost', 'host_listings_count', 'neighbourhood_cleansed', 
    'neighbourhood_group_cleansed', 'latitude', 'longitude', 'room_type', 'accommodates', 'price_quote_checkin_date', 'price_quote_checkout_date',
    'price_quote_price_per_night', 'minimum_nights', 'number_of_reviews', 'number_of_reviews_l30d', 'review_scores_rating', 
    'reviews_per_month', 'availability_30', 'calendar_last_scraped', 'calculated_host_listings_count', 'estimated_occupancy_l365d', 
    'estimated_revenue_l365d']].copy()

df_analysis


,id,name,host_id,host_name,host_is_superhost,host_listings_count,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,...,minimum_nights,number_of_reviews,number_of_reviews_l30d,review_scores_rating,reviews_per_month,availability_30,calendar_last_scraped,calculated_host_listings_count,estimated_occupancy_l365d,estimated_revenue_l365d
1,1215930899551057941,Queens apartament,392517036,Roman,f,1.0,Maspeth,Queens,40.721708,-73.907840,...,31.0,0,0,NaN,NaN,29,2026-09-14,1,0,0.0
7,791539646807027059,The studio in the heights,493039547,Manuel,f,1.0,Washington Heights,Manhattan,40.854703,-73.938423,...,30.0,0,0,NaN,NaN,30,2026-09-14,1,0,0.0
8,984905638210475309,Beach vibes in New York City,538113856,Isabel,f,1.0,Maspeth,Queens,40.726240,-73.905540,...,30.0,0,0,NaN,NaN,30,2026-09-14,1,0,0.0
9,42220146,nice privet room,291026998,Ghenadie,f,4.0,Sheepshead Bay,Brooklyn,40.600260,-73.957930,...,30.0,0,0,NaN,NaN,29,2026-09-14,4,0,0.0
10,42220001,NICE AND COZY STUDIO,291026998,Ghenadie,f,4.0,Sheepshead Bay,Brooklyn,40.600350,-73.959570,...,30.0,0,0,NaN,NaN,29,2026-09-14,4,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96151,1641475769215499069,#2B-Comfortable Shared Apartment | Fulton st,535892414,Noor,f,75.0,Bedford-Stuyvesant,Brooklyn,40.678630,-73.908290,...,30.0,0,0,NaN,NaN,30,2026-03-16,15,0,0.0
96152,1641477591868154681,#2D- Cozy and Convinent stay at Fulton st Broo...,535892414,Noor,f,75.0,Bedford-Stuyvesant,Brooklyn,40.677120,-73.909570,...,30.0,0,0,NaN,NaN,30,2026-03-16,15,0,0.0
96153,1641479712612774109,#2C- Cozy And Comfortable stay at Fulton st,535892414,Noor,f,75.0,Bedford-Stuyvesant,Brooklyn,40.676970,-73.909690,...,30.0,0,0,NaN,NaN,30,2026-03-16,15,0,0.0
96154,1641520475827435951,"55""tv, exposed brick, AC, Prime Williamsburg #636",586470023,Michael,f,120.0,Williamsburg,Brooklyn,40.720835,-73.961054,...,30.0,0,0,NaN,NaN,30,2026-03-16,115,0,0.0


In [43]:
df_analysis.loc[df_analysis['id'].duplicated()].sort_values('id').head(10)

,id,name,host_id,host_name,host_is_superhost,host_listings_count,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,...,minimum_nights,number_of_reviews,number_of_reviews_l30d,review_scores_rating,reviews_per_month,availability_30,calendar_last_scraped,calculated_host_listings_count,estimated_occupancy_l365d,estimated_revenue_l365d
29918,2539,11 Min to Manhattan • Prospect Park • Fast WiFi,2787,John,f,5.0,Kensington,Brooklyn,40.645937,-73.972164,...,30.0,10,1,4.80,0.08,4,2026-06-15,5,60,6838.0
29919,6848,Only 2 stops to Manhattan studio,15991,Allen,t,1.0,Williamsburg,Brooklyn,40.709350,-73.953420,...,30.0,198,0,4.60,0.95,1,2026-06-14,1,180,21109.0
60184,6848,Only 2 stops to Manhattan studio,15991,Allen,t,1.0,Williamsburg,Brooklyn,40.709350,-73.953420,...,30.0,197,0,4.60,0.96,5,2026-03-16,1,240,28145.0
60185,6872,Uptown Sanctuary w/ Private Bath (Month to Month),16104,Kae,f,2.0,East Harlem,Manhattan,40.801070,-73.942550,...,30.0,2,0,5.00,0.04,23,2026-03-16,2,60,4019.0
29921,6990,UES Beautiful Blue Room,16800,Cynthia,t,1.0,East Harlem,Manhattan,40.787780,-73.947590,...,30.0,251,0,4.88,1.24,0,2026-06-14,1,120,9260.0
60186,6990,UES Beautiful Blue Room,16800,Cynthia,t,1.0,East Harlem,Manhattan,40.787780,-73.947590,...,30.0,251,0,4.88,1.26,0,2026-03-16,1,240,18521.0
29922,7097,"Perfect for Your Parents, With Garden & Patio",17571,Jane,t,2.0,Fort Greene,Brooklyn,40.691940,-73.973890,...,30.0,423,0,4.89,2.12,0,2026-06-14,2,255,51630.0
60188,7801,Sunny Williamsburg Loft with Private Cedar Sauna,21207,Chaya,t,3.0,Williamsburg,Brooklyn,40.718807,-73.956177,...,30.0,12,0,4.91,0.06,0,2026-03-16,1,0,0.0
29923,7801,Sunny Williamsburg Loft with Private Cedar Sauna,21207,Chaya,t,3.0,Williamsburg,Brooklyn,40.718807,-73.956177,...,30.0,13,0,4.92,0.06,0,2026-06-14,1,60,21958.0
60189,8490,"Maison des Sirenes1,bohemian, luminous apartment",25183,Nathalie,f,2.0,Bedford-Stuyvesant,Brooklyn,40.684556,-73.939634,...,30.0,189,0,4.77,0.94,9,2026-03-16,2,0,0.0


In [44]:
df_analysis.to_csv('../data/processed/listings_combined.csv', index=False)